<a href="https://colab.research.google.com/github/BF667-IDLE/vsep/blob/main/notebooks/vsep_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 vsep — Audio Stem Separator

Welcome to the **vsep** interactive demo! Separate any audio into vocals, drums, bass, instruments, and more using 100+ state-of-the-art AI models.

## Features
- ⚡ **Fast Processing** — GPU-accelerated inference with automatic hardware detection
- 🎯 **100+ Models** — Choose from Demucs, MDX-Net, VR Band Split, and Roformer architectures
- 🎚️ **Easy to Use** — Browse, select, separate, listen, and download in 6 steps
- 📖 **[Full Wiki](https://github.com/BF667-IDLE/vsep/wiki)** for detailed docs

---

## 1️⃣ Setup

Clone the repo and install dependencies. This takes ~2 minutes.

In [ ]:
#@title 1. Install vsep

!git clone -q https://github.com/BF667-IDLE/vsep.git /content/vsep
%cd /content/vsep
!pip install -q -r requirements.txt

import os
os.makedirs("output", exist_ok=True)
os.makedirs("models", exist_ok=True)

import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️ No GPU detected — will use CPU (slower)")
    print("💡 Go to Runtime → Change runtime type → T4 GPU")

## 2️⃣ Upload Your Audio

Upload any audio file. Supported: MP3, WAV, FLAC, OGG, M4A, AAC, WMA

In [ ]:
#@title 2. Upload Audio

from google.colab import files
import os

print("📁 Upload your audio file...")
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    size_mb = os.path.getsize(audio_file) / (1024 * 1024)
    print(f"✅ Uploaded: {audio_file} ({size_mb:.1f} MB)")
else:
    print("❌ No file uploaded.")

## 3️⃣ Browse Models

View all available models grouped by task. Copy the **display name** and paste it in the next cell.

In [ ]:
#@title 3. Browse Models

import sys
sys.path.insert(0, "/content/vsep")

from utils.cli import _print_model_categories, _print_model_table
from separator import Separator

separator = Separator(info_only=True)
model_list = separator.list_models()

cat_filter = "All" #@param ["All", "Vocals", "Instrumental", "Multi-Stem", "Reverb/Echo Cleanup", "Drums/Bass/Other", "Denoise/Cleanup"]

if cat_filter == "All":
    _print_model_categories(model_list)
else:
    cat_kw = {
        "Vocals": ["vocal", "karaoke", "crowd", "bleed", "fullness", "male-female", "bandit", "scnet", "revive", "resurrection", "ft", "instvoc", "viperx"],
        "Instrumental": ["inst", "instrumental", "exp ", "phantom", "hp2", "sp ", "mid ", "wind", "mgm", "duality"],
        "Multi-Stem": ["demucs", "6s", "male-female", "4-stem", "4s"],
        "Reverb/Echo Cleanup": ["reverb", "echo", "dereverb", "deecho"],
        "Drums/Bass/Other": ["drum", "bass", "other", "kuielab"],
        "Denoise/Cleanup": ["denoise", "d1581", "aspiration", "debleed", "noise"],
    }
    kws = cat_kw.get(cat_filter, [])
    filtered = {}
    for mtype, models in model_list.items():
        matching = {n: d for n, d in models.items() if any(k in n.lower() for k in kws)}
        if matching:
            filtered[mtype] = matching
    if filtered:
        total = sum(len(v) for v in filtered.values())
        print(f"\n  [{cat_filter}]  {total} models")
        _print_model_table(filtered)
    else:
        print(f"  No models found for [{cat_filter}]")

total = sum(len(v) for v in model_list.values())
print(f"\n  Total catalog: {total} models")
print("  ➡️ Copy the display name and paste it in the next cell.")

In [ ]:
#@title 3b. Select Model

model = "model_bs_roformer_ep_317_sdr_12.9755.ckpt" #@param {type:"string"}

if not model.strip():
    raise SystemExit("Enter a model name first.")

# Try to resolve display name to filename
resolved = None
for mtype, models in model_list.items():
    for name, data in models.items():
        if model.strip() == name:
            resolved = data["filename"]
            break
    if resolved:
        break

if resolved:
    selected_model = resolved
    print(f"  Display:  {model}")
    print(f"  File:     {resolved}")
elif model.endswith((".ckpt", ".onnx", ".pth", ".yaml")):
    selected_model = model.strip()
    print(f"  Custom model: {selected_model}")
else:
    # Fuzzy search
    matches = []
    for mtype, models in model_list.items():
        for name in models:
            if model.lower() in name.lower():
                matches.append((mtype, name))
    if matches:
        mtype, match_name = matches[0]
        selected_model = model_list[mtype][match_name]["filename"]
        print(f"  Did you mean: {match_name}")
        print(f"  File:       {selected_model}")
    else:
        raise SystemExit(f"Model not found: {model}. Run the Browse cell to see available models.")

print(f"\n  ✅ Ready to separate with: {selected_model}")

## 4️⃣ Separate Audio

Run the separation. First use may take a few minutes to download the model (cached afterwards).

In [ ]:
#@title 4. Run Separation

import sys
sys.path.insert(0, "/content/vsep")

from separator import Separator
import config.variables as cfg
import os, time

# Optimize for Colab
cfg.MAX_DOWNLOAD_WORKERS = 4
cfg.DOWNLOAD_CHUNK_SIZE = 262144

print("🚀 Starting separation...")
print(f"📁 Input: {audio_file}")
print(f"🎯 Model: {selected_model}")
print("⏳ Please wait...\n")

start = time.time()

# Initialize and run
separator = Separator(
    model_file_dir="./models",
    output_dir="./output",
    use_soundfile=True,
)

separator.load_model(selected_model)
output_files = separator.separate(audio_file)

elapsed = time.time() - start

print(f"\n✅ Separation complete! ({elapsed:.1f}s)")
print(f"📂 {len(output_files)} stem(s):")
for f in output_files:
    size = os.path.getsize(f) / (1024 * 1024)
    print(f"   — {os.path.basename(f)} ({size:.1f} MB)")

## 5️⃣ Listen to Results

Play back the separated stems.

In [ ]:
#@title 5. Listen to Stems

import glob
from IPython.display import Audio, display

output_files = sorted(glob.glob("output/*"))

if output_files:
    print(f"🎵 {len(output_files)} separated stem(s):\n")
    for i, fp in enumerate(output_files, 1):
        name = os.path.basename(fp)
        print(f"{i}. {name}")
        display(Audio(fp))
        print("─" * 60)
else:
    print("❌ No output files. Run the separation first.")

## 6️⃣ Download Results

Download the separated stems to your computer.

In [ ]:
#@title 6. Download Stems

from google.colab import files
import glob, os

output_files = sorted(glob.glob("output/*"))

if output_files:
    print(f"📥 Downloading {len(output_files)} file(s)...\n")
    for fp in output_files:
        files.download(fp)
else:
    print("❌ No files to download. Run the separation first.")

---

## 💡 Tips & Recommendations

### Best Models by Task
| Task | Recommended Model | Why |
|------|-----------------|-----|
| **Vocals** | `model_bs_roformer_ep_317_sdr_12.9755.ckpt` | Highest SDR (12.98) of any model |
| **4-stem** | `htdemucs_ft.yaml` | Vocals + drums + bass + other |
| **6-stem** | `htdemucs_6s.yaml` | Adds piano + guitar |
| **Instrumental** | `melband_roformer_inst_v2.ckpt` | Clean instrumental |
| **Karaoke** | `mel_band_roformer_karaoke_becruily.ckpt` | Aggressive vocal removal |
| **De-Reverb** | `dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt` | Remove reverb from vocals |
| **Denoise** | `denoise_mel_band_roformer_aufr33_sdr_27.9959.ckpt` | Clean up background noise |
| **Fast** | `UVR-MDX-NET-Inst_HQ_5.onnx` | Lightweight, fast inference |

### GPU Tips
- **T4 (free)** — Good for most models. Big Roformer/Demucs may be tight on 16 GB
- **A100 (paid)** — Can run any model including the largest ones
- **Timeouts** — Sessions timeout after ~90 min of inactivity
- **Long files** — For audio > 30 min, add `chunk_duration=300` in the separation cell

### Troubleshooting

| Problem | Fix |
|---------|-----|
| **No GPU** | Runtime → Change runtime type → T4 GPU |
| **Out of memory** | Use a smaller model or shorter audio, or add `chunk_duration=300` |
| **Download failed** | Re-run the cell (downloads resume automatically) |
| **Module not found** | Re-run the Install cell or restart the runtime |

### Links

- 📦 **Repository:** [github.com/BF667-IDLE/vsep](https://github.com/BF667-IDLE/vsep)
- 📖 **Wiki:** [github.com/BF667-IDLE/vsep/wiki](https://github.com/BF667-IDLE/vsep/wiki)
- 🐛 **Issues:** [github.com/BF667-IDLE/vsep/issues](https://github.com/BF667-IDLE/vsep/issues)
- 📄 **License:** [MIT](https://opensource.org/licenses/MIT)

**Made with ❤️ using [vsep](https://github.com/BF667-IDLE/vsep)**